# ERA5-Land Computational Overhead

This revision notebook measures wall-clock runtime and peak GPU memory for precipitation-related experiments only: vanilla ERA5-Land super-resolution, Flow Matching statistical-bias correction, and hypothesized-GEVD/heavier-tail precipitation. It does not run toy overhead experiments.

The notebook reruns timing pipelines with the same paper hyperparameters where they are defined in the original notebooks, but it does not save model checkpoints, generated samples, sample arrays, replacement figures, or intermediate tensors. Tables and small diagnostic plots are displayed in the notebook only.


## Complexity Reference

For reference, one standard ERM update costs approximately
\[
O(bC_{\rm fb}).
\]
A naive full-quantile differentiable update costs
\[
O((b+N_{\tilde x})C_{\rm fb}+N_{\tilde x}\log N_{\tilde x}).
\]
The IICT amortized cost is
\[
O((b+n_q)C_{\rm fb})+\frac1\omega O(N_{\tilde x}C_{\rm fwd}+N_{\tilde x}\log N_{\tilde x}).
\]
The practical memory estimate is
\[
O((b+n_q)\dim(u)).
\]


The following code imports the timing helpers, records hardware metadata, and sets up a runtime table. It prefers the current checkout as `WORK_DIR`, with a fallback to the server path used in the original notebooks.


In [ ]:
import gc
import os
import platform
import sys
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

WORK_DIR = Path.cwd()
SERVER_WORK_DIR = Path('/home/research/jenzheng/documents/kai/research/eta/eta')
if not (WORK_DIR / 'data/ERA5/era5land_USA_SouthEast_1999-2023_dailymax.nc').exists() and SERVER_WORK_DIR.exists():
    WORK_DIR = SERVER_WORK_DIR
sys.path.append(str(WORK_DIR))

from models import SRCNN, UNet
from revision_utils import (
    disable_checkpoint_saving,
    format_seconds,
    get_peak_vram,
    hardware_summary,
    measure_wall_time,
    predict_field_array,
    reset_peak_memory,
    run_without_saving,
    train_precip_eta_no_save,
    train_srcnn_mse_no_save,
)
from utils import global_seed

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def maybe_data_parallel(model):
    return nn.DataParallel(model) if torch.cuda.device_count() > 1 else model

hardware = hardware_summary(DEVICE)
try:
    import psutil
    hardware['system_ram_GB'] = psutil.virtual_memory().total / 1024**3
except Exception:
    hardware['system_ram_GB'] = np.nan

display(pd.DataFrame([hardware]))

runtime_rows = []

def append_runtime_row(
    experiment,
    stage,
    seconds,
    peak_memory,
    n_epochs=np.nan,
    batch_size=np.nan,
    n_quantiles=np.nan,
    tau=np.nan,
    omega=np.nan,
    N_aux=np.nan,
    n_generated_samples=np.nan,
    overhead_reference=None,
    overhead_ratio=np.nan,
    notes='',
):
    runtime_rows.append({
        'experiment': experiment,
        'stage': stage,
        'wall_time_seconds': float(seconds),
        'wall_time_hms': format_seconds(seconds),
        'peak_vram_allocated_GB': peak_memory.get('peak_vram_allocated_GB', np.nan),
        'peak_vram_reserved_GB': peak_memory.get('peak_vram_reserved_GB', np.nan),
        'device': str(DEVICE),
        'gpu_name': hardware.get('gpu_name'),
        'n_epochs': n_epochs,
        'batch_size': batch_size,
        'n_quantiles': n_quantiles,
        'tau': tau,
        'omega': omega,
        'N_aux': N_aux,
        'n_generated_samples': n_generated_samples,
        'overhead_reference': overhead_reference,
        'overhead_ratio': overhead_ratio,
        'notes': notes,
    })


## Shared ERA5-Land Data Preparation

The following cell mirrors the vanilla precipitation notebooks: load the 1999-2023 daily maximal precipitation fields, convert meters to millimeters, downsample by factor 10, trim the very largest held-out tail days used by the original notebooks, and construct the 0.5-year supervised split.


In [ ]:
import xarray as xr
from torch.utils.data import DataLoader, TensorDataset

file_path = WORK_DIR / 'data/ERA5/era5land_USA_SouthEast_1999-2023_dailymax.nc'
if not file_path.exists():
    raise FileNotFoundError(f'Missing ERA5-Land file: {file_path}')

ds = xr.open_dataset(file_path, engine='netcdf4')
tp = ds['tp']
tp_numpy = tp.values * 1000

ds_fact = 10
tp_ds_numpy = tp_numpy[:, ::ds_fact, ::ds_fact]

max_values_original = np.max(tp_numpy, axis=(1, 2))
sorted_indices_original = np.argsort(max_values_original)
sorted_max_values_original = max_values_original[sorted_indices_original]
trim_tail_thresh = 240
num_trim_days = len(sorted_max_values_original[sorted_max_values_original > trim_tail_thresh])
trim_days = sorted_indices_original[-num_trim_days:]
kept_days = np.delete(np.arange(len(tp_numpy)), trim_days)

tp_trim_numpy = tp_numpy[kept_days]
tp_trim_ds_numpy = tp_ds_numpy[kept_days]
tp_trim_tensor = torch.tensor(tp_trim_numpy, dtype=torch.float32).unsqueeze(1).to(DEVICE)
tp_trim_ds_tensor = torch.tensor(tp_trim_ds_numpy, dtype=torch.float32).unsqueeze(1).to(DEVICE)

num_years = 0.5
portion = num_years / 25
train_size = int(portion * len(tp_trim_ds_tensor))
train_input = tp_trim_ds_tensor[:train_size]
train_target = tp_trim_tensor[:train_size]
test_input = deepcopy(tp_trim_ds_tensor)
test_target = deepcopy(tp_trim_tensor)

batch_size = 64
train_dataset = TensorDataset(train_input, train_target)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataset = TensorDataset(test_input, test_target)
test_loader = DataLoader(test_dataset, batch_size=25 * batch_size, shuffle=False)

print('train_input:', train_input.shape)
print('train_target:', train_target.shape)
print('test_input:', test_input.shape)
print('test_target:', test_target.shape)


## Vanilla Precipitation Super-Resolution Overhead

This section measures the matched ERM/MSE SRCNN training time and η-training time with IICT. The paper hyperparameters used here are: SRCNN with hidden dimension 64, 3 residual blocks, scale factor 10, Adam learning rate \(3\times10^{-4}\), ERM training for 500 epochs, η training for 150 epochs, \(\lambda=1\), tail threshold 150 mm, and IICT refresh period \(\omega=30\).


The following code prepares the vanilla tail quantile/IICT objects and runs ERM training without saving checkpoints.


In [ ]:
global_seed(43)
max_values = np.max(tp_trim_numpy, axis=(1, 2))
sorted_indices = np.argsort(max_values)
sorted_max_values = max_values[sorted_indices]
w1_tail_thresh = 150
num_w1_days = len(sorted_max_values[sorted_max_values > w1_tail_thresh])
w1_truedays = sorted_indices[-num_w1_days:]
w1_truemax = torch.tensor(sorted_max_values[-num_w1_days:], dtype=torch.float32)

model_mse_timed = maybe_data_parallel(SRCNN(hidden_dim=64, num_blocks=3, scale_factor=ds_fact)).to(DEVICE)
_, erm_seconds, erm_peak = measure_wall_time(
    train_srcnn_mse_no_save,
    model_mse_timed,
    train_loader,
    test_loader,
    500,
    3e-4,
    scheduler_step_size=200,
    scheduler_gamma=0.2,
    device=DEVICE,
)
append_runtime_row(
    'Vanilla precipitation',
    'ERM training',
    erm_seconds,
    erm_peak,
    n_epochs=500,
    batch_size=batch_size,
    N_aux=len(test_input),
    notes='SRCNN MSE baseline; no checkpoint saved',
)
print('ERM training:', format_seconds(erm_seconds), erm_peak)


The following code initializes η training from the timed ERM model and measures the IICT η continuation. The overhead ratio is
\[
R_{\eta/{\rm ERM}}=\frac{T_\eta}{T_{\rm ERM}}.
\]


In [ ]:
global_seed(43)
model_eta_timed = deepcopy(model_mse_timed).to(DEVICE)
mse_input = train_input
mse_target = train_target
lambd_ = 1.0
omega = 30
varying_days = True

(_, _, full_output_eta_timed), eta_seconds, eta_peak = measure_wall_time(
    train_precip_eta_no_save,
    model_eta_timed,
    test_loader,
    mse_input,
    mse_target,
    tp_trim_ds_numpy,
    w1_truemax,
    w1_truedays,
    150,
    3e-4,
    lambd_,
    omega,
    varying_days,
    seed=43,
    device=DEVICE,
)
append_runtime_row(
    'Vanilla precipitation',
    'η training with IICT',
    eta_seconds,
    eta_peak,
    n_epochs=150,
    batch_size=batch_size,
    n_quantiles=num_w1_days,
    omega=omega,
    N_aux=len(test_input),
    overhead_reference='Vanilla precipitation / ERM training',
    overhead_ratio=eta_seconds / erm_seconds,
    notes='Initialized from ERM model; trained in memory; no checkpoint saved',
)
print('η training:', format_seconds(eta_seconds), eta_peak)


## Flow Matching Statistical-Bias-Correction Overhead

This section measures three distinct Flow Matching costs: training, sampling, and η-map pass-through. The LR Flow Matching model uses the full 25-year LR data with 80 epochs and 100 ODE steps, matching the LR sample file used by the plotting notebook. The HR Flow Matching timing row uses the 0.5-year HR baseline with 200 epochs and 200 ODE steps, matching the scarce-HR baseline in the plotting notebook. Both use the U-Net velocity field with `n_channels=16`, `ch_mults=(1,2,2,4)`, `is_attn=(False,False,True,True)`, `n_blocks=1`, and Adam learning rate \(3\times10^{-4}\).


The following code defines the no-save Flow Matching preprocessing, training, and Dormand-Prince sampling helpers used only by this timing notebook.


In [ ]:
def get_hires_dgm_train(num_years):
    portion = num_years / 25
    train_size = int(portion * len(tp_trim_ds_tensor))
    dgm_train = np.clip(tp_trim_numpy[:train_size], 1e0, None)
    dgm_train = np.log(dgm_train)
    train_mean = dgm_train.mean()
    train_std = dgm_train.std()
    normalized = (dgm_train - train_mean) / train_std
    inverse_transform = lambda x: np.exp(x * train_std + train_mean)
    return normalized, inverse_transform


def get_lores_dgm_train(num_years):
    portion = num_years / 25
    train_size = int(portion * len(tp_trim_ds_tensor))
    dgm_train = np.clip(tp_trim_ds_numpy[:train_size], 1e0, None)
    dgm_train = np.log(dgm_train)
    train_mean = dgm_train.mean()
    train_std = dgm_train.std()
    normalized = (dgm_train - train_mean) / train_std
    inverse_transform = lambda x: np.exp(x * train_std + train_mean)
    return normalized, inverse_transform


def dopri5_step(v, x, t, h):
    k1 = v(x, t)
    k2 = v(x + h * (1 / 5) * k1, t + h / 5)
    k3 = v(x + h * (3 / 40 * k1 + 9 / 40 * k2), t + 3 * h / 10)
    k4 = v(x + h * (44 / 45 * k1 - 56 / 15 * k2 + 32 / 9 * k3), t + 4 * h / 5)
    k5 = v(x + h * (19372 / 6561 * k1 - 25360 / 2187 * k2 + 64448 / 6561 * k3 - 212 / 729 * k4), t + 8 * h / 9)
    k6 = v(x + h * (9017 / 3168 * k1 - 355 / 33 * k2 + 46732 / 5247 * k3 + 49 / 176 * k4 - 5103 / 18656 * k5), t + h)
    return x + h * (35 / 384 * k1 + 500 / 1113 * k3 + 125 / 192 * k4 - 2187 / 6784 * k5 + 11 / 84 * k6)


def dopri5_fm_sampling(model, step_size, nsamples, batch_size, image_size):
    H, W, C = image_size
    num_steps = int(1.0 / step_size)
    samples_list = []
    n_batches = nsamples // batch_size + (1 if nsamples % batch_size else 0)
    model.eval()
    with torch.no_grad():
        for batch_idx in range(n_batches):
            current_batch_size = batch_size if batch_idx < n_batches - 1 else nsamples - batch_size * batch_idx
            x = torch.randn(current_batch_size, C, H, W, device=DEVICE)
            t = torch.zeros(x.shape[0], device=DEVICE)
            for _ in range(num_steps):
                x = dopri5_step(model, x, t, step_size)
                t += step_size
            samples_list.append(x.detach().cpu())
    return torch.cat(samples_list, dim=0)[:nsamples]


The following code trains the HR and LR Flow Matching velocity fields in memory only, with checkpoint saving disabled by construction.


In [ ]:
from flow_matching.path import AffineProbPath
from flow_matching.path.scheduler import CondOTScheduler

image_channels = 1
n_channels = 16
ch_mults = (1, 2, 2, 4)
is_attn = (False, False, True, True)
n_blocks = 1
fm_lr = 3e-4

def build_fm_model():
    model = UNet(image_channels=image_channels, n_channels=n_channels,
                 ch_mults=ch_mults, is_attn=is_attn, n_blocks=n_blocks)
    return maybe_data_parallel(model).to(DEVICE)


def train_fm_no_save(train_array, epochs, train_batch_size):
    model = build_fm_model()
    path = AffineProbPath(scheduler=CondOTScheduler())
    optimizer = torch.optim.Adam(model.parameters(), lr=fm_lr)
    cfm_loss = nn.MSELoss()
    fm_train = torch.tensor(train_array, dtype=torch.float32).unsqueeze(1)
    dataset = TensorDataset(fm_train)
    train_loader_fm = DataLoader(dataset, batch_size=train_batch_size, shuffle=True, num_workers=1, pin_memory=True)
    model.train()
    for _ in range(epochs):
        for (x_1,) in train_loader_fm:
            optimizer.zero_grad()
            x_1 = x_1.to(DEVICE)
            x_0 = torch.randn_like(x_1).to(DEVICE)
            t = torch.rand(x_1.shape[0]).to(DEVICE)
            path_sample = path.sample(t=t, x_0=x_0, x_1=x_1)
            loss = cfm_loss(model(path_sample.x_t, path_sample.t), path_sample.dx_t)
            loss.backward()
            optimizer.step()
    return model

hr_fm_train, hr_inverse_transform = get_hires_dgm_train(0.5)
lr_fm_train, lr_inverse_transform = get_lores_dgm_train(25)

hr_fm_model, hr_train_seconds, hr_train_peak = measure_wall_time(
    train_fm_no_save,
    hr_fm_train,
    200,
    128,
    device=DEVICE,
)
append_runtime_row(
    'Flow Matching',
    'HR FM training',
    hr_train_seconds,
    hr_train_peak,
    n_epochs=200,
    batch_size=128,
    notes='0.5-year HR FM baseline; trained in memory; no checkpoint saved',
)
print('HR FM training:', format_seconds(hr_train_seconds), hr_train_peak)

lr_fm_model, lr_train_seconds, lr_train_peak = measure_wall_time(
    train_fm_no_save,
    lr_fm_train,
    80,
    128,
    device=DEVICE,
)
append_runtime_row(
    'Flow Matching',
    'LR FM training',
    lr_train_seconds,
    lr_train_peak,
    n_epochs=80,
    batch_size=128,
    notes='25-year LR FM model; trained in memory; no checkpoint saved',
)
print('LR FM training:', format_seconds(lr_train_seconds), lr_train_peak)


The following code measures HR and LR Flow Matching sampling separately. It uses 9044 generated samples, with 200 ODE steps for the HR baseline and 100 ODE steps for the LR sample generator. Generated samples are kept in memory only until the timing-dependent downstream step is complete.


In [ ]:
nsamples = 9044

hr_latent_samples, hr_sample_seconds, hr_sample_peak = measure_wall_time(
    dopri5_fm_sampling,
    hr_fm_model,
    1 / 200,
    nsamples,
    9044,
    (80, 160, 1),
    device=DEVICE,
)
append_runtime_row(
    'Flow Matching',
    'HR sampling',
    hr_sample_seconds,
    hr_sample_peak,
    batch_size=9044,
    n_generated_samples=nsamples,
    notes='Dormand-Prince 5(4), 200 steps; samples not saved',
)
hr_samples = hr_inverse_transform(hr_latent_samples.squeeze().numpy())
del hr_latent_samples, hr_samples
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('HR FM sampling:', format_seconds(hr_sample_seconds), hr_sample_peak)

lr_latent_samples, lr_sample_seconds, lr_sample_peak = measure_wall_time(
    dopri5_fm_sampling,
    lr_fm_model,
    1 / 100,
    nsamples,
    9044,
    (8, 16, 1),
    device=DEVICE,
)
append_runtime_row(
    'Flow Matching',
    'LR sampling',
    lr_sample_seconds,
    lr_sample_peak,
    batch_size=9044,
    n_generated_samples=nsamples,
    notes='Dormand-Prince 5(4), 100 steps; samples not saved',
)
lr_samples = lr_inverse_transform(lr_latent_samples.squeeze().numpy())
del lr_latent_samples
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('LR FM sampling:', format_seconds(lr_sample_seconds), lr_sample_peak)


The following code measures the η-map correction/pass-through time after LR samples have been generated in memory. This is reported separately from Flow Matching training and sampling:
\[
T_{\eta{\rm -pass}}.
\]


In [ ]:
eta_pass_dataset = TensorDataset(torch.tensor(lr_samples, dtype=torch.float32).unsqueeze(1))
eta_pass_loader = DataLoader(eta_pass_dataset, batch_size=256, shuffle=False)
eta_corrected_samples, eta_pass_seconds, eta_pass_peak = measure_wall_time(
    predict_field_array,
    model_eta_timed,
    eta_pass_loader,
    device=DEVICE,
)
append_runtime_row(
    'Flow Matching',
    'η correction pass',
    eta_pass_seconds,
    eta_pass_peak,
    batch_size=256,
    n_generated_samples=nsamples,
    notes='Pass generated LR samples through timed vanilla η map; corrected samples not saved',
)
print('η correction pass:', format_seconds(eta_pass_seconds), eta_pass_peak)
del lr_samples, eta_corrected_samples, eta_pass_dataset, eta_pass_loader
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Hypothesized-GEVD Precipitation Overhead

This section measures η training under the hypothesized heavier-tailed GEVD prior. It uses the same SRCNN downscaler, 0.5-year supervised split, Adam learning rate \(3\times10^{-4}\), \(\lambda=1\), \(\tau=0.95\), \(n_q=350\), \(\omega=1\), and 150 η epochs as `ERA5Land-EVD.ipynb`. The comparable overhead ratio is
\[
R_{{\rm GEVD}\eta/{\rm ERM}}=\frac{T_{{\rm GEVD}\eta}}{T_{\rm ERM}}.
\]


The following code fits the GEVD prior as in the GEVD notebook, constructs the 350 reference quantiles, and measures in-memory η training without checkpoint saving.


In [ ]:
from scipy.integrate import quad
from scipy.stats import genextreme


def fit_gev_with_cutoff(data, cutoff):
    filtered_data = data[data <= cutoff]
    shape, loc, scale = genextreme.fit(filtered_data)
    scale = scale + 4
    normalization_constant = quad(
        lambda t: genextreme.pdf(t, shape, loc=loc, scale=scale),
        -np.inf,
        cutoff,
    )[0]
    return {
        'shape': shape,
        'location': loc,
        'scale': scale,
        'cutoff': cutoff,
        'c': normalization_constant,
    }

fit_results = fit_gev_with_cutoff(max_values, cutoff=np.ceil(max(max_values)) + 20)
tau_gevd = 0.95
Qy_gevd = lambda q: genextreme.ppf(q * fit_results['c'], fit_results['shape'], fit_results['location'], fit_results['scale'])
num_w1_days_gevd = 350
quantiles_gevd = torch.linspace(tau_gevd, 1, num_w1_days_gevd)
w1_truemax_gevd = torch.tensor([Qy_gevd(float(q)) for q in quantiles_gevd], dtype=torch.float32)
w1_truedays_gevd = sorted_indices[-num_w1_days_gevd:]

model_gevd_eta_timed = deepcopy(model_mse_timed).to(DEVICE)
(_, _, full_output_gevd_eta_timed), gevd_eta_seconds, gevd_eta_peak = measure_wall_time(
    train_precip_eta_no_save,
    model_gevd_eta_timed,
    test_loader,
    mse_input,
    mse_target,
    tp_trim_ds_numpy,
    w1_truemax_gevd,
    w1_truedays_gevd,
    150,
    3e-4,
    1.0,
    1,
    True,
    seed=43,
    device=DEVICE,
)
append_runtime_row(
    'GEVD precipitation',
    'η training',
    gevd_eta_seconds,
    gevd_eta_peak,
    n_epochs=150,
    batch_size=batch_size,
    n_quantiles=num_w1_days_gevd,
    tau=tau_gevd,
    omega=1,
    N_aux=len(test_input),
    overhead_reference='Vanilla precipitation / ERM training',
    overhead_ratio=gevd_eta_seconds / erm_seconds,
    notes='Hypothesized GEVD prior; trained in memory; no checkpoint saved',
)
print('GEVD η training:', format_seconds(gevd_eta_seconds), gevd_eta_peak)


## Runtime Summary

The following cell displays the timing/VRAM summary table and small notebook-visible bar plots. No table or plot is saved to disk.


In [ ]:
runtime_summary_df = pd.DataFrame(runtime_rows)
display(runtime_summary_df)

if len(runtime_summary_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), dpi=150)
    labels = runtime_summary_df['experiment'] + ' / ' + runtime_summary_df['stage']
    axes[0].barh(labels, runtime_summary_df['wall_time_seconds'])
    axes[0].set_xlabel('Wall time (seconds)')
    axes[0].set_title('Runtime by stage')
    axes[0].grid(axis='x', alpha=0.3)

    axes[1].barh(labels, runtime_summary_df['peak_vram_allocated_GB'])
    axes[1].set_xlabel('Peak allocated VRAM (GB)')
    axes[1].set_title('Peak GPU memory by stage')
    axes[1].grid(axis='x', alpha=0.3)

    fig.tight_layout()
    plt.show()


# Revision documentation notes

This notebook corresponds to the precipitation-related runtime and peak-memory measurements requested during revision. It covers vanilla ERA5-Land super-resolution, Flow Matching statistical-bias correction, and hypothesized-GEVD precipitation. It reports ERM training, η training with IICT, Flow Matching HR/LR training, Flow Matching HR/LR sampling, η-map pass-through, and GEVD η training as separate rows. No runtime table is saved for cross-notebook use; results are displayed directly in this notebook. The timing cells rerun models and generated samples in memory only, with no new checkpoints, generated sample arrays, or figures written to disk. Existing experiment notebooks were not modified to perform these measurements.
